### Tables of units and blades' map.

In [13]:
AMPSUB = {
    0    : 1.0,    # no unit defined.
    "mA" : 1e-3,   # mili
    "uA" : 1e-6,   # micro
    "nA" : 1e-9,   # nano
    "pA" : 1e-12,  # pico
    "fA" : 1e-15,  # femto
    "aA" : 1e-18,  # atto
}

# The XBPM beamlines.
BEAMLINENAME = {
    "CAT": "Cateretê",
    "CNB": "Carnaúba",
    "MGN": "Mogno",
    "MNC": "Manacá",
}

BLADEMAP = {
    "MNC": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CAT":  {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    # "CAT1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},

    # ## To be checked: ## #
    # "CAT2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CNB": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MGN1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "SIM":  {"TO": 'A', "TI": 'B', "BI": 'C', "BO": 'D'},
}

FILE_EXTENSION = ".pickle"    # Data file type.

### Procedures for 2025-06-11

#### Set of functions to read data from files and restructure them for each beamline, in columns for each blade's value, the undulator gap and the SR current.

In [7]:
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle  # noqa: S403


In [ ]:
# Functions for opening, rearranging and plotting data.

def data_from_file(wdir):
    """Read data from pickle files."""
    allfiles = os.listdir(wdir)
    picklefiles = [pf for pf in allfiles if pf.endswith("pickle")]
    sfiles = sorted(picklefiles, key=lambda name: lastfield(name, '_'))

    rawdata = list()
    for file in sfiles:
        with open(wdir + "/" + file, 'rb') as df:
            rawdata.append(pickle.load(df))  # noqa: S301

    return rawdata


def lastfield(name, fld=' '):  # noqa: D103
    return name.split(fld)[-1]


def read_rawdata(rawdata):
    """."""
    data = dict()
    blk = 0
    for nline in range(6):
        beamline = rawdata[blk][0][nline]['name']
        data[beamline] = dict()

    for blk in range(len(rawdata)):
        for nline, bline in enumerate(data.keys()):
            data[bline][blk] = dict()
            # For each block.
            data[bline][blk]['data'] = deepcopy(rawdata[blk][0][nline])
            data[bline][blk]['info'] = deepcopy(rawdata[blk][1])

    return data


def data_structure_average(rdata):
    """Average over blades' values and simplify data structure."""
    data = dict()
    for bline in rdata.keys():
        for blk in range(len(rdata[bline].keys())):
            for bld in ["A", "B", "C", "D"]:
                blade = f"{bld}_val"
                av = average_blade(rdata[bline][blk]['data'][blade])
                rdata[bline][blk]['data'][blade] = av
                del rdata[bline][blk]['data'][f"{bld}_range"]
            del rdata[bline][blk]['data']["name"]
            del rdata[bline][blk]['data']["prefix"]

            rdata[bline][blk]['data']["current"] = \
                np.round(rdata[bline][blk]['info']["current"], decimals=-1)
            del rdata[bline][blk]['info']["current"]

            bl = bline[:3].lower()   # Gap info key.
            if bl in rdata[bline][blk]['info'].keys():
                rdata[bline][blk]['data']['gap'] = \
                    np.round(rdata[bline][blk]['info'][bl], decimals=0)

            del rdata[bline][blk]["info"]
            rdata[bline][blk] = rdata[bline][blk]["data"]

        dt = dict()
        for key, rd in rdata[bline].items():
            dt[key] = rd
        data[bline] = dt
    return data


def average_blade(blade):
    """Function for averaging over each blade's measurement."""
    bld = []
    for val in blade:
        bld.append(val[0] * AMPSUB[val[1]])
    return np.array([np.average(bld), np.std(bld)])


def pandas_data_frame(data):
    """Data to pandas' data frame."""
    pdata = dict()
    for key, val in data.items():
        pdata[key] = pd.DataFrame(val)
    return pdata


def blades_data_array(pdata, beamline):
    """Divide data into arrays for each blade."""
    blades = {
        "A_val" : [],
        "B_val" : [],
        "C_val" : [],
        "D_val" : [],
    }

    for ip, p in enumerate(pdata[beamline]):
        crr = np.round(pdata[beamline][p]["current"], decimals=-1)

        try:
            if beamline in ["CAT", "CNB"]:
                gap = np.round(pdata[beamline][p]["gap"], decimals=0)
            elif beamline in ["MNC1", "MNC2"]:
                gap = pdata[beamline][p]["gap"]
            else:
                gap = None  # pdata[beamline][p]["gap"]
        except Exception as err:
            print(f" WARNING: beamline {beamline}, data # {ip}:"
                  f"\t Exception when trying to define {err}")
            gap = None

        # print(f">>> (BLADES DATA RRAY) gap {beamline} = {gap}")

        for bl in blades.keys():
            blval = pdata[beamline][p][bl]
            # print(f" {cr} ({type(cr)}) : {bl} → {blval}")
            blades[bl].append([crr, gap, blval[0], blval[1]])

    for key, val in blades.items():
        blades[key] = np.array(val)

    return blades


def blades_plot(blades, ax0, ax1, beamline):
    """Plot data."""
    # Plot markers for each blade.
    markers = {'A_val' : 'o',
               'B_val' : '^',
               'C_val' : 's',
               'D_val' : '*'}

    for key, val in blades.items():
        x = val[:, 0]
        y = val[:, 2]
        s = val[:, 3]

        # beamlines have different amperimeters.
        if beamline in ["CAT", "CNB"]:
            ylabel = u"$I$ [nA]"
            y *= 1e9     # Current in nA.
        else:
            ylabel = "counts"

        mrk = markers[key]
        mask = val[:, 2] >= -5e-3
        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            # Beamlines with undulator source.
            gaps = np.unique(val[:, 1])
            # Divide in groups by undulator's gap.
            for gap in gaps:
                mask_gap = val[:, 1] == gap
                gmask = mask & mask_gap
                ax0.errorbar(x[gmask], y[gmask], s[gmask],
                            fmt=f'{mrk}-', label=key)
                ax1.errorbar(x[gmask], y[gmask], s[gmask],
                            fmt=f'{mrk}-', label=f"{key} @ gap {gap}")

        else:
            # Beamline with dipole source.
            ax0.errorbar(x[mask], y[mask], s[mask],
                         fmt=f'{mrk}-', label=key)

    title_curr = u"XBPM $\\times$ SR currents @"
    ax0.set_title(f"{title_curr} {beamline}")
    ax0.set_xlabel("SR current [mA]")
    ax0.set_ylabel(ylabel)
    ax0.legend()
    ax0.grid()

    if ax1 is not None:
        title_gap = u"XBPM $\\times$ undulator gaps @"
        ax1.set_title(f"{title_gap} {beamline}")
        ax1.set_xlabel("SR current [mA]")
        ax1.set_ylabel(ylabel)
        ax1.legend()
        ax1.grid()


In [9]:
%matplotlib qt5

## Main.

In [21]:
def main(wdir, outdir):
    """."""
    # List with data read from working directory.
    rawdata = data_from_file(wdir)

    # Extract data from read files and compose a dict
    # relative to each beam line.
    rdata   = read_rawdata(rawdata)

    # Average over blades' measurements and
    # simplify data structure dictionary.
    data    = data_structure_average(rdata)

    # Export data to a pandas data frame.
    pdata   = pandas_data_frame(data)

    for beamline in pdata.keys():

        # Divide pdata into arrays for each blade.
        blades = blades_data_array(pdata, beamline=beamline)

        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            fig, (axi, axg) = plt.subplots(1, 2, figsize=(20, 9))
            blades_plot(blades, axi, axg, beamline)
        else:
            fig, axi = plt.subplots(1, 1, figsize=(10, 6))
            blades_plot(blades, axi, None, beamline)

        fig.tight_layout()
        fig.savefig(f"{outdir}/XBPM_{beamline}_SR_current.png")

    plt.show()


basedir = "/home/arnaldo.filho/XBPM/medidas/results_2025/"
# wdir = "MNC_1_20250729/subsec_09SA"
# wdir = "MNC_1_20250623/2025-06-23"

# wdir = basedir + "Gaps_20250616/2025-06-16"
outdir = basedir + "Gaps_20250616/"
wdir = outdir + "2025-06-16"

# wdir = basedir + "MNC_1_20250623/2025-06-23"

if __name__ == "__main__":
    main(wdir, outdir)

In [220]:
wdir = "/home/arnaldo.filho/XBPM/medidas/results_2025/Gaps_20250616/2025-06-16"

rawdata = data_from_file(wdir)

# print(f"\n KEYS = {rawdata[0][0][0].keys()} \n")

# Check info.
# for ii in range(19):
#     for rd in rawdata[ii][1]:
#         # print(f"({rd['name']:5}) {rd.keys()}")
#         print(f"{rd} = {rawdata[ii][1][rd]:14.4f}", end="\t")
#     print()

# Check data keys.
# for ii in range(19):
#     for rd in rawdata[ii][0]:
#         print(f"{rd.keys()}")
#     print()

rdata = read_rawdata(rawdata)
# rdata['CNB'][0]['data']["C_val"]
data   = data_structure_average(rdata)
pdata  = pandas_data_frame(data)

# gaps = np.unique(blades["A_val"][:, 1])

# for gap in gaps:
#     print(f" Gap = {gap}")
#     mask = blades["A_val"][:, 1] == gap
#     print(f" I = {blades['A_val'][mask, 3]}")


pdata["MNC2"].T
# blades
# data

# beamline = "MGN2"
# blades = blades_data_array(pdata, beamline)
# blades

,A_val,B_val,C_val,D_val,current,gap
0,"[36.5, 0.5]","[29.3, 0.45825756949558394]","[-327.5, 0.5]","[-67.9, 0.7000000000000001]",-0.0,11.0
1,"[35.6, 0.4898979485566356]","[29.1, 0.3]","[-329.3, 0.7810249675906654]","[-67.9, 0.7000000000000001]",-0.0,11.0
2,"[1510.1, 0.5385164807134505]","[1067.4, 0.48989794855663565]","[2087.8, 0.9797958971132713]","[443.9, 0.7]",20.0,11.0
3,"[5227.1, 0.5385164807134505]","[5704.1, 0.3]","[3202.1, 0.5385164807134505]","[21821.0, 1.2649110640673518]",20.0,7.0
4,"[13437.2, 1.9390719429665315]","[15652.5, 1.5]","[6684.0, 1.3416407864998738]","[61160.8, 3.026549190084311]",20.0,0.0
5,"[32080.2, 1.4000000000000001]","[37234.6, 1.1135528725660042]","[15616.3, 1.268857754044952]","[145212.3, 4.075536774462966]",50.0,0.0
6,"[12087.5, 4.883646178829912]","[13689.6, 3.104834939252005]","[7747.9, 3.6728735344413916]","[52139.7, 7.416872656315464]",50.0,7
7,"[12087.5, 4.883646178829912]","[13689.6, 3.104834939252005]","[7747.9, 3.6728735344413916]","[52139.7, 7.416872656315464]",50.0,11.0
8,"[7444.2, 0.6]","[4886.2, 0.39999999999999997]","[11491.7, 0.9000000000000001]","[2395.6, 0.66332495807108]",100.0,11.0
9,"[23462.2, 0.9797958971132711]","[27088.7, 1.3453624047073711]","[15253.9, 1.6401219466856725]","[102325.2, 4.48998886412873]",100.0,7.0
